# ⚗️ Gayatri Chemistry Tutor — Fine-Tuning Qwen2.5-3B-Instruct (QLoRA)

This notebook fine-tunes **`Qwen/Qwen2.5-3B-Instruct`** (3 Billion parameters) on the Gayatri Chemistry curriculum dataset.
It uses **4-bit QLoRA (Quantized Low-Rank Adaptation)** with `bitsandbytes`, making it 100% compatible with Google Colab's free T4 GPU without running Out of Memory (OOM).

**Complete Workflow:**
1. Mount Google Drive for automatic backup
2. Install pinned, compatible training stack (`transformers`, `peft`, `trl`, `bitsandbytes`, `accelerate`)
3. Upload `train.jsonl` and `validation.jsonl`
4. Load `Qwen/Qwen2.5-3B-Instruct` in 4-bit (NF4) and configure LoRA adapters ($r=16, \alpha=32$)
5. Fine-tune using `SFTTrainer` with gradient accumulation & fp16
6. Socratic pedagogical verification test (Inorganic VSEPR / $\text{NH}_3$ geometry)
7. Save LoRA adapter, reload base model in fp16, cleanly merge weights
8. Convert merged model to GGUF using `llama.cpp` and quantize to `Q4_K_M` (`Gayatri-Tutor-v3-Q4_K_M.gguf`)
9. Automatically copy the final GGUF model into Google Drive (`/MyDrive/GayatriAI/models/gayatri/`) and trigger browser download


## 1. Environment Setup & Google Drive Mount

In [ ]:
# 1. Mount Google Drive for automatic backup
from google.colab import drive
import os
import shutil

print("[*] Mounting Google Drive...")
drive.mount('/content/drive')

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/GayatriAI/models/gayatri/"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"[OK] Google Drive backup directory ready: {DRIVE_BACKUP_DIR}")

# 2. Remove Colab's pre-installed torchao and install pinned, mutually-compatible training stack
!pip uninstall -y -q torchao
!pip install --quiet \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "trl==0.12.2" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "bitsandbytes>=0.43.0" \
    "huggingface_hub==0.26.2"

import torch
print(f"[OK] PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("[ERROR] No GPU detected! Please go to Runtime > Change runtime type > T4 GPU before running.")
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[OK] GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")


## 2. Upload Training Dataset (`train.jsonl` and `validation.jsonl`)

In [ ]:
from google.colab import files

required_files = ["train.jsonl", "validation.jsonl"]
missing = [f for f in required_files if not os.path.exists(f)]

if missing:
    print(f"Please upload the missing file(s) {missing} from PRIVATE_WORK/training/exported/:")
    uploaded = files.upload()
    still_missing = [f for f in required_files if not os.path.exists(f)]
    if still_missing:
        raise FileNotFoundError(
            f"Still missing {still_missing} after upload. "
            "Make sure the filenames match exactly (case-sensitive)."
        )

print("[OK] train.jsonl and validation.jsonl are verified and ready!")


## 3. Load Qwen2.5-3B in 4-bit (QLoRA) & Configure LoRA Adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
print(f"[*] Loading base model and tokenizer: {BASE_MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit NF4 Quantization (allows 3B model to load in ~2.2 GB VRAM on Colab T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## 4. Fine-Tune on Gayatri Chemistry Curriculum

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files={"train": "train.jsonl", "validation": "validation.jsonl"})
print(f"[*] Training samples: {len(dataset['train'])}, Validation: {len(dataset['validation'])}")

def format_chatml(example):
    messages = example["messages"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

formatted_dataset = dataset.map(format_chatml)

# SFTConfig calibrated specifically for 3B QLoRA on free Colab T4
training_args = SFTConfig(
    output_dir="./gayatri_3b_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,          # Batch size 2 fits comfortably in 15GB VRAM
    gradient_accumulation_steps=4,         # Effective batch size = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,                             # T4 fp16 tensor cores acceleration
    bf16=False,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    tokenizer=tokenizer,
    args=training_args,
)

print("[*] Fine-tuning Qwen2.5-3B started...")
trainer.train()
print("[OK] Fine-tuning complete!")


## 5. Verification Test (Sample Pedagogical Inference)

In [ ]:
prompt_msgs = [
    {"role": "system", "content": "You are Gayatri, an expert adaptive NCERT chemistry tutor teaching senior secondary chemistry."},
    {"role": "user", "content": "Let's switch to Chemical Bonding. Why does NH3 have a pyramidal shape instead of tetrahedral or trigonal planar?"}
]
formatted_prompt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.2,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )
print("\n--- Generated Pedagogical Response ---")
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))


## 6. Save LoRA Adapters & Cleanly Merge into 16-bit Hugging Face Model

In [ ]:
import gc
from peft import PeftModel

ADAPTER_DIR = "./gayatri_3b_adapter"
MERGED_DIR = "./gayatri_3b_merged"

print("[*] Saving trained LoRA adapter weights...")
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Free training VRAM before loading full fp16 model for merge
print("[*] Freeing GPU memory...")
del model
del trainer
torch.cuda.empty_cache()
gc.collect()

print(f"[*] Reloading base {BASE_MODEL_NAME} in float16 for clean merge...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

print("[*] Merging LoRA weights with base model...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"[OK] LoRA weights merged cleanly into 16-bit model at: {MERGED_DIR}")

# Free memory before GGUF conversion
del base_model
del merged_model
torch.cuda.empty_cache()
gc.collect()


## 7. Convert to GGUF and Quantize to Q4_K_M

In [ ]:
# 1. Install build tools & fetch llama.cpp
!apt-get -qq update && apt-get -qq install -y cmake build-essential > /dev/null
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
!pip install --quiet gguf
!pip install --quiet -r llama.cpp/requirements.txt
!cd llama.cpp && cmake -B build && cmake --build build --config Release -j --target llama-quantize


In [ ]:
# 2. Convert merged model to f16 GGUF
!python llama.cpp/convert_hf_to_gguf.py ./gayatri_3b_merged --outfile ./gayatri_f16.gguf --outtype f16

assert os.path.exists("./gayatri_f16.gguf"), "f16 GGUF conversion failed — check the log above."
print(f"[OK] Unquantized f16 GGUF ready: {os.path.getsize('./gayatri_f16.gguf') / 1e9:.2f} GB")


In [ ]:
# 3. Quantize to Q4_K_M
GGUF_NAME = "Gayatri-Tutor-v3-Q4_K_M.gguf"

quant_bin_candidates = [
    "./llama.cpp/build/bin/llama-quantize",
    "./llama.cpp/llama-quantize",
]
quant_bin = next((p for p in quant_bin_candidates if os.path.exists(p)), None)
if quant_bin is None:
    raise FileNotFoundError(
        "Could not find the llama-quantize binary. Check that the cmake build step above succeeded."
    )

!{quant_bin} ./gayatri_f16.gguf {GGUF_NAME} Q4_K_M

assert os.path.exists(GGUF_NAME), "Quantization failed — check the log above."
print(f"[OK] Quantized model ready: {GGUF_NAME} ({os.path.getsize(GGUF_NAME) / 1e6:.1f} MB)")


## 8. Backup to Google Drive & Direct Download

In [ ]:
# 1. Save directly to Google Drive
drive_dest = os.path.join(DRIVE_BACKUP_DIR, GGUF_NAME)
shutil.copy(GGUF_NAME, drive_dest)
print(f"\n✅ SUCCESS! Model backed up to Google Drive at: {drive_dest}")

# 2. Trigger direct browser download
from google.colab import files
print("[*] Triggering direct download to your local machine...")
files.download(GGUF_NAME)
